In [1]:
%load_ext autoreload
%autoreload 2

# Lesson Learn

## StateMachine
This idea using statemachine idea to iterative run until we met the end state

## State: with Enum
This idea serves as a blueprint what we can move to which state

## Data: with pydantic.BaseModel
This idea serves as a data spec that will be used in each state

## Method: with ABC @abstractmethod
This idea serves as a contract of each class

# New State Machine Idea

In [2]:
from broinsight.statemachines.simple_statemachine import SimpleStateMachine, simple_state, BaseSimpleContext, BaseSimpleState, SimpleStateRegistry

In [3]:
from broinsight.core.llm import LocalOpenAI, UserMessage, AIMessage
from broinsight.prompt_hub import PromptHub

In [4]:
print(PromptHub().quick_chat)

# PERSONA

You're Andy who is the best bro in the world. You're always chill and supportive.

# INSTRUCTION

- understand {USER_INPUT}
- always respond your message in bro-like manner

# CONDITION

- your answer will always be concise
- if {USER_INPUT} has the intent indicating a user needs a long, detailed answer like: `explain in detail`, `explain it step by step`, you can answer it accordingly



In [5]:
from enum import Enum
from typing import Any
from pydantic import Field

class SimpleState(Enum):
    USER_INPUT = "user_input"
    ROUTER = "router"
    CHAT = "chat"
    COMPLETE = "complete"

class SimpleContext(BaseSimpleContext):
    user_input:Any = Field(default=None)
    input_state:Any = Field(default=None)
    output_state:Any = Field(default=None)
    messages:list = Field(default_factory=list)

@simple_state(SimpleState.USER_INPUT)
class UserInputState(BaseSimpleState):
    def next_state(self): return SimpleState.ROUTER
    def run(self, context: SimpleContext):
        context.user_input = input("Enter your input: ")
        return self.next_state()

@simple_state(SimpleState.ROUTER)
class RouterState(BaseSimpleState):
    def next_state(self, user_input:str):
        if user_input.lower().startswith("/exit"):
            return SimpleState.COMPLETE
        return SimpleState.CHAT
    def run(self, context: SimpleContext): return self.next_state(context.user_input)
    
@simple_state(SimpleState.CHAT)
class ChatState(BaseSimpleState):
    def __init__(self): self.llm = LocalOpenAI()
    def next_state(self): return SimpleState.USER_INPUT
    def run(self, context: SimpleContext):
        messages = context.messages
        messages.append(UserMessage(content=context.user_input))
        response = self.llm.run(PromptHub().quick_chat, messages)
        messages.append(AIMessage(content=response.content))
        context.messages = messages
        return self.next_state()

In [6]:
SimpleStateRegistry._states

{'USER_INPUT': __main__.UserInputState,
 'ROUTER': __main__.RouterState,
 'CHAT': __main__.ChatState}

In [7]:
SimpleStateRegistry.state_graph()

{'USER_INPUT': ['ROUTER'],
 'ROUTER': ['CHAT', 'COMPLETE'],
 'CHAT': ['USER_INPUT']}

In [8]:
SimpleStateRegistry.to_mermaid(save_path="./flow.md", direction="TB")

'flowchart TB\n    USER_INPUT --> ROUTER\n    ROUTER -.-> CHAT\n    ROUTER -.-> COMPLETE\n    CHAT --> USER_INPUT'

In [11]:
state_machine = SimpleStateMachine(SimpleState.USER_INPUT, SimpleState.COMPLETE)
result = state_machine.run(SimpleContext())

In [12]:
result.execution_trace

[{'state': <SimpleState.USER_INPUT: 'user_input'>,
  'next_state': <SimpleState.ROUTER: 'router'>},
 {'state': <SimpleState.ROUTER: 'router'>,
  'next_state': <SimpleState.CHAT: 'chat'>},
 {'state': <SimpleState.CHAT: 'chat'>,
  'next_state': <SimpleState.USER_INPUT: 'user_input'>},
 {'state': <SimpleState.USER_INPUT: 'user_input'>,
  'next_state': <SimpleState.ROUTER: 'router'>},
 {'state': <SimpleState.ROUTER: 'router'>,
  'next_state': <SimpleState.COMPLETE: 'complete'>}]

In [13]:
result.messages

[{'role': 'user', 'content': 'Hi'},
 {'role': 'assistant', 'content': "Hey bro! What's good?"}]

In [14]:
SimpleStateRegistry.clear_all_states()

In [14]:
SimpleStateRegistry._states

{'USER_INPUT': __main__.UserInputState,
 'ROUTER': __main__.RouterState,
 'CHAT': __main__.ChatState}

In [1]:
import seaborn as sns
import duckdb
# tips = sns.load_dataset('tips')
duckdb.register('tips', sns.load_dataset('tips'))

In [8]:
duckdb.execute("DESCRIBE tips;").df()

,column_name,column_type,null,key,default,extra
0,total_bill,DOUBLE,YES,None,None,None
1,tip,DOUBLE,YES,None,None,None
2,sex,"ENUM('Male', 'Female')",YES,None,None,None
3,smoker,"ENUM('Yes', 'No')",YES,None,None,None
4,day,"ENUM('Thur', 'Fri', 'Sat', 'Sun')",YES,None,None,None
5,time,"ENUM('Lunch', 'Dinner')",YES,None,None,None
6,size,BIGINT,YES,None,None,None


In [5]:
duckdb.execute("DESCRIBE tips;").fetchall()

[('total_bill', 'DOUBLE', 'YES', None, None, None),
 ('tip', 'DOUBLE', 'YES', None, None, None),
 ('sex', "ENUM('Male', 'Female')", 'YES', None, None, None),
 ('smoker', "ENUM('Yes', 'No')", 'YES', None, None, None),
 ('day', "ENUM('Thur', 'Fri', 'Sat', 'Sun')", 'YES', None, None, None),
 ('time', "ENUM('Lunch', 'Dinner')", 'YES', None, None, None),
 ('size', 'BIGINT', 'YES', None, None, None)]

In [7]:
duckdb.description()

[('column_name', 'STRING', None, None, None, None, None),
 ('column_type', 'STRING', None, None, None, None, None),
 ('null', 'STRING', None, None, None, None, None),
 ('key', 'STRING', None, None, None, None, None),
 ('default', 'STRING', None, None, None, None, None),
 ('extra', 'STRING', None, None, None, None, None)]

In [11]:
# Show all tables
result = duckdb.execute("SHOW TABLES").fetchall()
print(f"Number of tables: {len(result)}")

# Or get count directly
count = duckdb.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'main'").fetchall()
print(f"Table count: {count}")


Number of tables: 1
Table count: [(1,)]


In [12]:
result

[('tips',)]

In [13]:
count

[(1,)]